# IMDb data enrichment: film synopses and principal biographies

For every feature-length film released between 1980 and 2025, this notebook takes the **top 10 films of each genre for each year** and enriches them with:

- a **film synopsis** (the article's *Plot* section, plus the article's introduction as a short summary);
- **biographical information for at least three principals** of each film (Wikipedia introduction, plus birth date, death date and birthplace).

**Run `IMDb_Data_Quality_and_Exploration.ipynb` first**, with the same `MAX_ROWS` value: this notebook reuses its cleaned Spark tables.

## Sources, rules and lineage

| Information | Source |
|---|---|
| IMDb ID → Wikidata item → English Wikipedia article | Wikidata property P345 (IMDb ID), via the SPARQL endpoint |
| Birth date, death date, birthplace | Wikidata (P569, P570, P19) |
| Synopsis (Plot section) and film summary (introduction) | English Wikipedia article wikitext, latest revision |
| Biography | English Wikipedia article introduction (TextExtracts API) |

- Human-authored open sources only: Wikipedia and Wikidata are written by human editors. No text is generated by a model.
- Single source family: If a film or person has no English Wikipedia article, no other source is searched.
- Matching by ID, not by title: Wikidata's IMDb-ID property links the IMDb record to the right article, so there's no guessing between films that share a title.
- Rate limits and policies: Requests are sent one at a time with at least `REQUEST_INTERVAL_SECONDS` between them, with a descriptive User-Agent since its mandatory, `maxlag=5` and `Retry-After` honoured on HTTP 429/503. Requests are batched (200 IDs per SPARQL query, 10–20 titles per API call) to keep their number low.
- Lineage: Every enriched value carries the URL it came from: the Wikipedia revision URL, the Wikidata item URL, the retrieval time and the licence. Each film record also has a `lineage` array listing every source URL used for it.
- Spark does the processing: Only the downloads run outside Spark (`wikimedia_client.py`). Reading the raw files with an enforced schema, cleaning, deduplication, wikitext-to-text conversion, joins, principal selection, notes and output are Spark operations.


## 1. Setup

In [1]:
%pip -q install mwparserfromhell==0.6.6

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import os
from pathlib import Path

import pandas as pd
from IPython.display import display
from pyspark.sql import SparkSession, Window, functions as F, types as T

import wikimedia_client as wc

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 200)

SPARK_DATA_DIR = Path(os.environ.get("SPARK_DATA_DIR", "spark_data"))
MAX_ROWS = int(os.environ.get("MAX_ROWS", "3000"))
RUN_NAME = "full" if MAX_ROWS < 0 else f"sample_{MAX_ROWS}"
WORK_DIR = SPARK_DATA_DIR / RUN_NAME
OUTPUT_DIR = Path("results") / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SPARK_DRIVER_MEMORY = os.environ.get("SPARK_DRIVER_MEMORY", "4g")

YEAR_FROM, YEAR_TO = 1980, 2025
FEATURE_MIN_RUNTIME = 70
TOP_N_PER_GENRE_YEAR = 10
PRINCIPALS_PER_FILM = 3
PRINCIPAL_CANDIDATES = 6
ENRICH_FILM_LIMIT = int(os.environ.get("ENRICH_FILM_LIMIT", "0"))
REQUEST_INTERVAL_SECONDS = 1.0
RAW_DIR = Path("enrichment_raw")

spark = (
    SparkSession.builder
    .appName("imdb-data-enrichment")
    .master("local[*]")
    .config("spark.driver.memory", SPARK_DRIVER_MEMORY)
    .config("spark.sql.shuffle.partitions", "64")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.ansi.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Run mode: {RUN_NAME} (MAX_ROWS = {MAX_ROWS}), film limit: {ENRICH_FILM_LIMIT or 'none'}, results in {OUTPUT_DIR}/")


def export(df, filename, n=20):
    pdf = df.toPandas()
    display(pdf.head(n))
    pdf.to_csv(OUTPUT_DIR / filename, index=False)
    return pdf


def save_table(name, df):
    path = str(WORK_DIR / "enrichment" / name)
    df.write.mode("overwrite").parquet(path)
    spark.read.parquet(path).createOrReplaceTempView(name)
    return spark.table(name)

Run mode: sample_3000 (MAX_ROWS = 3000), film limit: none, results in results/sample_3000/


## 2. Load the cleaned IMDb tables

These Parquet tables are written in `IMDb_Data_Quality_and_Exploration.ipynb`.

In [3]:
for table in ["titles_clean", "ratings_clean", "principals_clean", "people_clean"]:
    path = WORK_DIR / "clean" / table
    if not (path / "_SUCCESS").exists():
        raise FileNotFoundError(
            f"{path} is missing. Run IMDb_Data_Quality_and_Exploration.ipynb first with MAX_ROWS = {MAX_ROWS}."
        )
    spark.read.parquet(str(path)).createOrReplaceTempView(table)
print("Cleaned tables loaded.")

Cleaned tables loaded.


## 3. Select the films to enrich

Feature-length films (`titleType = 'movie'`, runtime ≥ 70 minutes) released between 1980 and 2025 are ranked inside each (year, genre) by number of IMDb votes, then rating. The first `TOP_N_PER_GENRE_YEAR` of each group are kept. A film that is top 10 in several genres is enriched once.

In [4]:
rank_filter = f"WHERE genre_year_rank <= {TOP_N_PER_GENRE_YEAR}" if TOP_N_PER_GENRE_YEAR else ""
top_films_per_genre_year = save_table("top_films_per_genre_year", spark.sql(f'''
WITH films AS (
    SELECT t.tconst, t.primaryTitle, t.originalTitle, t.startYear, t.runtimeMinutes, t.genres,
           r.averageRating, r.numVotes
    FROM titles_clean t
    JOIN ratings_clean r ON r.tconst = t.tconst
    WHERE t.titleType = 'movie' AND t.runtimeMinutes >= {FEATURE_MIN_RUNTIME}
      AND t.startYear BETWEEN {YEAR_FROM} AND {YEAR_TO} AND t.genres IS NOT NULL
),
ranked AS (
    SELECT f.*, g.genre,
           ROW_NUMBER() OVER (PARTITION BY f.startYear, g.genre
                              ORDER BY f.numVotes DESC, f.averageRating DESC, f.tconst) AS genre_year_rank
    FROM films f
    LATERAL VIEW explode(split(f.genres, ',')) g AS genre
)
SELECT * FROM ranked {rank_filter}
'''))
export(  # Top 10 feature-length films per genre for each year 1980-2025
    top_films_per_genre_year.orderBy("startYear", "genre", "genre_year_rank"),
    "top_films_per_genre_year.csv",
)

films_df = spark.sql('''
SELECT tconst, FIRST(primaryTitle) AS primaryTitle, FIRST(originalTitle) AS originalTitle,
       FIRST(startYear) AS startYear, FIRST(runtimeMinutes) AS runtimeMinutes,
       FIRST(averageRating) AS averageRating, FIRST(numVotes) AS numVotes,
       split(FIRST(genres), ',') AS genres,
       sort_array(collect_list(named_struct('genre', genre, 'rank', genre_year_rank))) AS genre_year_ranks
FROM top_films_per_genre_year
GROUP BY tconst
''')
if ENRICH_FILM_LIMIT:
    films_df = films_df.orderBy(F.desc("numVotes"), "tconst").limit(ENRICH_FILM_LIMIT)
films_to_enrich = save_table("films_to_enrich", films_df)

principal_candidates = save_table("principal_candidates", spark.sql(f'''
SELECT * FROM (
    SELECT p.tconst, p.nconst, p.ordering, p.category, p.characters, n.primaryName,
           n.birthYear AS imdb_birthYear, n.deathYear AS imdb_deathYear,
           ROW_NUMBER() OVER (PARTITION BY p.tconst ORDER BY p.ordering, p.nconst) AS candidate_rank
    FROM principals_clean p
    LEFT SEMI JOIN films_to_enrich f ON f.tconst = p.tconst
    JOIN people_clean n ON n.nconst = p.nconst
    WHERE p.category NOT IN ('archive_footage', 'archive_sound')
) x
WHERE candidate_rank <= {PRINCIPAL_CANDIDATES}
'''))

print(f"Films to enrich: {films_to_enrich.count():,}; principal candidates: {principal_candidates.count():,} "
      f"({principal_candidates.select('nconst').distinct().count():,} distinct people)")

,tconst,primaryTitle,originalTitle,startYear,runtimeMinutes,genres,averageRating,numVotes,genre,genre_year_rank
0,tt0081573,Superman II,Superman II,1980,127,"Action,Adventure,Sci-Fi",6.8,125392,Action,1
1,tt0080684,Star Wars: Episode V - The Empire Strikes Back,Star Wars: Episode V - The Empire Strikes Back,1980,124,"Adventure,Fantasy,Sci-Fi",8.7,1533246,Adventure,1
2,tt0080455,The Blues Brothers,The Blues Brothers,1980,133,"Adventure,Comedy,Crime",7.9,232983,Adventure,2
3,tt0081573,Superman II,Superman II,1980,127,"Action,Adventure,Sci-Fi",6.8,125392,Adventure,3
4,tt0081398,Raging Bull,Raging Bull,1980,129,"Biography,Drama,Sport",8.1,416431,Biography,1
5,tt0080678,The Elephant Man,The Elephant Man,1980,124,"Biography,Drama",8.2,286316,Biography,2
6,tt0080339,Airplane!,Airplane!,1980,88,Comedy,7.7,294356,Comedy,1
7,tt0080455,The Blues Brothers,The Blues Brothers,1980,133,"Adventure,Comedy,Crime",7.9,232983,Comedy,2
8,tt0080487,Caddyshack,Caddyshack,1980,98,"Comedy,Sport",7.2,134990,Comedy,3
9,tt0080455,The Blues Brothers,The Blues Brothers,1980,133,"Adventure,Comedy,Crime",7.9,232983,Crime,1


Films to enrich: 2,090; principal candidates: 12,540 (5,547 distinct people)


## 4. Gather raw data from Wikidata and Wikipedia (outside Spark)


In [5]:
for name in ["wikidata_films", "wikidata_people", "wikipedia_films", "wikipedia_people"]:
    (RAW_DIR / f"{name}.jsonl").parent.mkdir(parents=True, exist_ok=True)
    (RAW_DIR / f"{name}.jsonl").touch()

film_ids = [r.tconst for r in films_to_enrich.select("tconst").collect()]
person_ids = [r.nconst for r in principal_candidates.select("nconst").distinct().collect()]

client = wc.RateLimitedSession(min_interval=REQUEST_INTERVAL_SECONDS)
stores = {
    "wikidata_films": wc.JsonlStore(RAW_DIR / "wikidata_films.jsonl", key="imdb_id"),
    "wikidata_people": wc.JsonlStore(RAW_DIR / "wikidata_people.jsonl", key="imdb_id"),
    "wikipedia_films": wc.JsonlStore(RAW_DIR / "wikipedia_films.jsonl", key="wikipedia_url"),
    "wikipedia_people": wc.JsonlStore(RAW_DIR / "wikipedia_people.jsonl", key="wikipedia_url"),
}

wc.collect(stores["wikidata_films"], film_ids, lambda ids: wc.wikidata_by_imdb_ids(client, ids), 200, "Wikidata films")
wc.collect(stores["wikidata_people"], person_ids,
           lambda ids: wc.wikidata_by_imdb_ids(client, ids, with_person_facts=True), 200, "Wikidata people")


def article_urls(path, ids):
    wanted, urls = set(ids), set()
    with open(path, encoding="utf-8") as handle:
        for line in handle:
            record = json.loads(line)
            if record["imdb_id"] in wanted and record.get("wikipedia_url"):
                urls.add(record["wikipedia_url"])
    return sorted(urls)


wc.collect(stores["wikipedia_films"], article_urls(RAW_DIR / "wikidata_films.jsonl", film_ids),
           lambda urls: wc.wikipedia_film_pages(client, urls), 10, "Wikipedia films")
wc.collect(stores["wikipedia_people"], article_urls(RAW_DIR / "wikidata_people.jsonl", person_ids),
           lambda urls: wc.wikipedia_person_intros(client, urls), 20, "Wikipedia people")
print(f"HTTP requests sent in this run: {client.request_count:,}")

Wikidata films: 2,090 requested, 0 already cached, 2,090 to fetch


  Wikidata films: 2,000/2,090


Wikidata people: 5,547 requested, 0 already cached, 5,547 to fetch


  Wikidata people: 2,000/5,547


  Wikidata people: 4,000/5,547


Wikipedia films: 2,091 requested, 0 already cached, 2,091 to fetch


... (41 outputs left out)

  Wikipedia people: 4,200/4,697


  Wikipedia people: 4,400/4,697


  Wikipedia people: 4,600/4,697


HTTP requests sent in this run: 484


## 5. Read the raw data in Spark with an enforced schema

Each landing file is read with an explicit schema. Lines that don't match it are kept in a `_corrupt_record` column and counted, instead of silently breaking the pipeline.

In [6]:
WIKIDATA_SCHEMA = T.StructType([
    T.StructField("imdb_id", T.StringType()),
    T.StructField("wikidata_id", T.StringType()),
    T.StructField("wikidata_url", T.StringType()),
    T.StructField("wikipedia_url", T.StringType()),
    T.StructField("wikidata_description", T.StringType()),
    T.StructField("birth_date", T.StringType()),
    T.StructField("death_date", T.StringType()),
    T.StructField("birth_place", T.StringType()),
    T.StructField("matches", T.IntegerType()),
    T.StructField("retrieved_at", T.TimestampType()),
    T.StructField("source", T.StringType()),
    T.StructField("license", T.StringType()),
    T.StructField("_corrupt_record", T.StringType()),
])
PAGE_FIELDS = [
    T.StructField("wikipedia_url", T.StringType()),
    T.StructField("resolved_title", T.StringType()),
    T.StructField("page_id", T.LongType()),
    T.StructField("revision_id", T.LongType()),
    T.StructField("revision_url", T.StringType()),
    T.StructField("missing", T.BooleanType()),
    T.StructField("retrieved_at", T.TimestampType()),
    T.StructField("source", T.StringType()),
    T.StructField("license", T.StringType()),
]
FILM_PAGE_SCHEMA = T.StructType(PAGE_FIELDS + [
    T.StructField("lead_wikitext", T.StringType()),
    T.StructField("plot_heading", T.StringType()),
    T.StructField("plot_wikitext", T.StringType()),
    T.StructField("_corrupt_record", T.StringType()),
])
PERSON_PAGE_SCHEMA = T.StructType(PAGE_FIELDS + [
    T.StructField("intro_text", T.StringType()),
    T.StructField("_corrupt_record", T.StringType()),
])


def read_raw(name, schema):
    return (spark.read.schema(schema)
            .option("mode", "PERMISSIVE").option("columnNameOfCorruptRecord", "_corrupt_record")
            .json(str(RAW_DIR / f"{name}.jsonl"))
            .cache())


raw = {
    "wikidata_films": read_raw("wikidata_films", WIKIDATA_SCHEMA),
    "wikidata_people": read_raw("wikidata_people", WIKIDATA_SCHEMA),
    "wikipedia_films": read_raw("wikipedia_films", FILM_PAGE_SCHEMA),
    "wikipedia_people": read_raw("wikipedia_people", PERSON_PAGE_SCHEMA),
}
raw_summary = pd.DataFrame([  # Raw records gathered per source and records rejected by the schema
    {"source_file": f"{name}.jsonl", "records": df.count(),
     "rejected_by_schema": df.where(F.col("_corrupt_record").isNotNull()).count()}
    for name, df in raw.items()
])
display(raw_summary)

,source_file,records,rejected_by_schema
0,wikidata_films.jsonl,2091,0
1,wikidata_people.jsonl,5715,0
2,wikipedia_films.jsonl,2091,0
3,wikipedia_people.jsonl,4697,0


## 6. Clean the raw data in Spark

- **Deduplication:** one row per IMDb ID and per article URL. A rerun or a Wikidata ID shared by several items can create several rows, so the row with an English article and the most recent retrieval wins.
- **Validation:** article URLs must point to English Wikipedia. Dates are converted from Wikidata's `YYYY-MM-DDT00:00:00Z` format to `DATE`. Dates before year 1 or with unknown precision become NULL.
- **Wikitext to plain text:** a Spark UDF using `mwparserfromhell` removes templates (infoboxes, citations), references, files, categories, HTML comments and markup. Spark functions then remove citation markers like `[12]`, empty brackets and extra whitespace.

In [7]:
@F.udf(T.StringType())
def wikitext_to_text(wikitext):
    if not wikitext:
        return None
    import mwparserfromhell

    code = mwparserfromhell.parse(wikitext)
    for node in code.filter_tags(matches=lambda n: str(n.tag).lower() in ("ref", "gallery", "math", "table")):
        try:
            code.remove(node)
        except ValueError:
            pass
    for link in code.filter_wikilinks():
        if str(link.title).strip().lower().startswith(("file:", "image:", "category:")):
            try:
                code.remove(link)
            except ValueError:
                pass
    return code.strip_code(normalize=True, collapse=True)


def tidy_text(column):
    text = F.regexp_replace(column, r"\[(\d+|citation needed|note \d+)\]", "")
    text = F.regexp_replace(text, r"\(\s*[;,]?\s*\)", "")
    text = F.regexp_replace(text, r"\(\s*[;,]\s*", "(")
    text = F.regexp_replace(text, r"[ \t ]+", " ")
    text = F.regexp_replace(text, r" *\n *", "\n")
    text = F.regexp_replace(text, r"\n{3,}", "\n\n")
    return F.when(F.trim(text) != "", F.trim(text))


def wikidata_date(column):
    return F.when(F.col(column).rlike(r"^\d{4}-\d{2}-\d{2}T"), F.to_date(F.substring(column, 1, 10)))


def latest_per_key(df, key, *priority):
    window = F.row_number().over(Window.partitionBy(key).orderBy(*priority, F.desc("retrieved_at")))
    return df.where(F.col("_corrupt_record").isNull()).withColumn("_rn", window).where("_rn = 1").drop("_rn", "_corrupt_record")


EN_WIKI = r"^https://en\.wikipedia\.org/wiki/.+"
wikidata_clean = {
    kind: save_table(f"wikidata_{kind}_clean", latest_per_key(
        raw[f"wikidata_{kind}"].withColumn(
            "wikipedia_url", F.when(F.col("wikipedia_url").rlike(EN_WIKI), F.col("wikipedia_url"))
        ),
        "imdb_id", F.col("wikipedia_url").isNull(), F.col("birth_date").isNull(),
    ).select(
        "imdb_id", "wikidata_id", "wikidata_url", "wikipedia_url", "matches",
        tidy_text(F.col("wikidata_description")).alias("wikidata_description"),
        wikidata_date("birth_date").alias("birth_date"),
        wikidata_date("death_date").alias("death_date"),
        tidy_text(F.col("birth_place")).alias("birth_place"),
        F.col("retrieved_at").alias("wikidata_retrieved_at"),
    ))
    for kind in ["films", "people"]
}

film_pages_clean = save_table("wikipedia_films_clean", latest_per_key(raw["wikipedia_films"], "wikipedia_url").select(
    "wikipedia_url", "resolved_title", "revision_id", "revision_url", "missing", "plot_heading",
    tidy_text(wikitext_to_text("plot_wikitext")).alias("synopsis"),
    tidy_text(wikitext_to_text("lead_wikitext")).alias("summary"),
    F.col("retrieved_at").alias("wikipedia_retrieved_at"), F.col("license").alias("wikipedia_license"),
))
person_pages_clean = save_table("wikipedia_people_clean", latest_per_key(raw["wikipedia_people"], "wikipedia_url").select(
    "wikipedia_url", "resolved_title", "revision_id", "revision_url", "missing",
    tidy_text(F.col("intro_text")).alias("biography"),
    F.col("retrieved_at").alias("wikipedia_retrieved_at"), F.col("license").alias("wikipedia_license"),
))
print("Cleaned:", {name: spark.table(name).count() for name in [
    "wikidata_films_clean", "wikidata_people_clean", "wikipedia_films_clean", "wikipedia_people_clean"]})

Cleaned: {'wikidata_films_clean': 2090, 'wikidata_people_clean': 5547, 'wikipedia_films_clean': 2091, 'wikipedia_people_clean': 4697}


## 7. Join, select three principals per film and add notes

For each film, principals are ranked by IMDb billing order (`ordering`). Principals with a Wikipedia biography come first, so the film gets at least `PRINCIPALS_PER_FILM` biographies whenever they exist among its first `PRINCIPAL_CANDIDATES` principals. When fewer exist, the remaining principals are kept with a note explaining what is missing.

Notes are added programmatically, at film and at principal level, whenever something is unavailable: no Wikidata item, no English article, no Plot section, too few principals with a biography.

In [8]:
spark.sql('''
SELECT c.*,
       w.wikidata_id, w.wikidata_url, w.wikipedia_url, w.birth_date, w.death_date, w.birth_place,
       w.wikidata_description,
       p.biography, p.revision_url AS biography_source_url, p.wikipedia_license,
       p.biography IS NOT NULL AS has_biography,
       filter(array(
           CASE WHEN w.imdb_id IS NULL THEN 'Wikidata lookup not run for this person' END,
           CASE WHEN w.imdb_id IS NOT NULL AND w.wikidata_id IS NULL THEN 'No Wikidata item has this IMDb ID' END,
           CASE WHEN w.wikidata_id IS NOT NULL AND w.wikipedia_url IS NULL THEN 'No English Wikipedia article' END,
           CASE WHEN w.wikipedia_url IS NOT NULL AND (p.missing OR p.biography IS NULL) THEN 'Wikipedia article found but its introduction is empty or missing' END,
           CASE WHEN w.matches > 1 THEN 'Several Wikidata items share this IMDb ID; the one with an English article was used' END
       ), x -> x IS NOT NULL) AS notes
FROM principal_candidates c
LEFT JOIN wikidata_people_clean w ON w.imdb_id = c.nconst
LEFT JOIN wikipedia_people_clean p ON p.wikipedia_url = w.wikipedia_url
''').createOrReplaceTempView("principal_details")

selected_principals = save_table("selected_principals", spark.sql(f'''
SELECT * FROM (
    SELECT d.*, ROW_NUMBER() OVER (PARTITION BY tconst ORDER BY has_biography DESC, ordering) AS pick_rank
    FROM principal_details d
) x
WHERE pick_rank <= {PRINCIPALS_PER_FILM}
'''))

enriched_films = save_table("enriched_films", spark.sql(f'''
WITH principals AS (
    SELECT tconst,
           sort_array(collect_list(named_struct(
               'ordering', ordering, 'nconst', nconst, 'name', primaryName, 'category', category,
               'characters', characters, 'imdb_birth_year', imdb_birthYear, 'imdb_death_year', imdb_deathYear,
               'biography', biography, 'birth_date', birth_date, 'death_date', death_date,
               'birth_place', birth_place, 'wikidata_description', wikidata_description,
               'biography_source_url', biography_source_url, 'wikidata_url', wikidata_url,
               'license', wikipedia_license, 'notes', notes
           ))) AS principals,
           SUM(CASE WHEN has_biography THEN 1 ELSE 0 END) AS principals_with_biography,
           COUNT(*) AS principals_listed
    FROM selected_principals
    GROUP BY tconst
)
SELECT f.tconst, f.primaryTitle, f.originalTitle, f.startYear, f.runtimeMinutes, f.genres,
       f.genre_year_ranks, f.averageRating, f.numVotes,
       w.wikidata_url, w.wikipedia_url, p.revision_url AS synopsis_source_url,
       p.plot_heading AS synopsis_section, p.synopsis, p.summary, p.wikipedia_license AS synopsis_license,
       p.wikipedia_retrieved_at AS synopsis_retrieved_at,
       COALESCE(pr.principals, array()) AS principals,
       COALESCE(pr.principals_with_biography, 0) AS principals_with_biography,
       filter(array(
           CASE WHEN w.imdb_id IS NULL THEN 'Wikidata lookup not run for this film' END,
           CASE WHEN w.imdb_id IS NOT NULL AND w.wikidata_id IS NULL THEN 'No Wikidata item has this IMDb ID' END,
           CASE WHEN w.wikidata_id IS NOT NULL AND w.wikipedia_url IS NULL THEN 'No English Wikipedia article; synopsis not available' END,
           CASE WHEN w.wikipedia_url IS NOT NULL AND (p.missing OR p.revision_id IS NULL) THEN 'Wikipedia article could not be retrieved' END,
           CASE WHEN p.revision_id IS NOT NULL AND p.synopsis IS NULL THEN 'Article has no Plot/Synopsis section; only the introduction is provided as summary' END,
           CASE WHEN p.synopsis IS NOT NULL AND length(p.synopsis) < 200 THEN 'Plot section is very short' END,
           CASE WHEN COALESCE(pr.principals_listed, 0) < {PRINCIPALS_PER_FILM} THEN concat('IMDb lists only ', COALESCE(pr.principals_listed, 0), ' usable principal(s) for this film') END,
           CASE WHEN COALESCE(pr.principals_with_biography, 0) < {PRINCIPALS_PER_FILM} THEN concat('Only ', COALESCE(pr.principals_with_biography, 0), ' of {PRINCIPALS_PER_FILM} selected principals have a Wikipedia biography') END
       ), x -> x IS NOT NULL) AS enrichment_notes,
       filter(array_distinct(concat(
           array(w.wikidata_url, p.revision_url),
           COALESCE(transform(pr.principals, x -> x.biography_source_url), array()),
           COALESCE(transform(pr.principals, x -> x.wikidata_url), array())
       )), x -> x IS NOT NULL) AS lineage
FROM films_to_enrich f
LEFT JOIN wikidata_films_clean w ON w.imdb_id = f.tconst
LEFT JOIN wikipedia_films_clean p ON p.wikipedia_url = w.wikipedia_url
LEFT JOIN principals pr ON pr.tconst = f.tconst
'''))
print(f"Enriched films: {enriched_films.count():,}")

Enriched films: 2,090


## 8. Output

In [9]:
with open(OUTPUT_DIR / "enriched_films.jsonl", "w", encoding="utf-8") as handle:  # Enriched films (one JSON record per film, with synopsis, principals, notes and lineage)
    for line in enriched_films.orderBy(F.desc("numVotes")).toJSON().toLocalIterator():
        handle.write(line + "\n")

enriched_principals = export(  # Flat table: one row per film and selected principal, with biography and source URLs
    spark.sql('''
    SELECT f.tconst, f.primaryTitle, f.startYear, f.synopsis_source_url,
           p.ordering, p.nconst, p.name, p.category, p.birth_date, p.death_date, p.birth_place,
           p.biography, p.biography_source_url, p.wikidata_url, concat_ws(' | ', p.notes) AS principal_notes
    FROM enriched_films f
    LATERAL VIEW explode(principals) e AS p
    ORDER BY f.numVotes DESC, f.tconst, p.ordering
    '''),
    "enriched_film_principals.csv",
)

enriched_films_flat = export(  # Flat table: one row per enriched film with synopsis, notes and lineage
    spark.sql('''
    SELECT tconst, primaryTitle, startYear, concat_ws(',', genres) AS genres, averageRating, numVotes,
           synopsis_section, synopsis, summary, synopsis_source_url, wikidata_url, principals_with_biography,
           concat_ws(' | ', enrichment_notes) AS enrichment_notes, concat_ws(' ', lineage) AS lineage
    FROM enriched_films
    ORDER BY numVotes DESC
    '''),
    "enriched_films.csv",
)

,tconst,primaryTitle,startYear,synopsis_source_url,ordering,nconst,name,category,birth_date,death_date,birth_place,biography,biography_source_url,wikidata_url,principal_notes
0,tt0111161,The Shawshank Redemption,1994,https://en.wikipedia.org/w/index.php?title=The_Shawshank_Redemption&oldid=1385870872,1,nm0000209,Tim Robbins,actor,1958-10-16,None,West Covina,"Timothy Francis Robbins (born October 16, 1958) is an American actor, director, producer, and writer. Known for his work on film and television, his accolades include an Academy Award, an Actor Aw...",https://en.wikipedia.org/w/index.php?title=Tim_Robbins&oldid=1384219365,http://www.wikidata.org/entity/Q95048,
1,tt0111161,The Shawshank Redemption,1994,https://en.wikipedia.org/w/index.php?title=The_Shawshank_Redemption&oldid=1385870872,2,nm0000151,Morgan Freeman,actor,1937-06-01,None,Memphis,"Morgan Freeman (born June 1, 1937) is an American actor, producer, and narrator. In a career spanning six decades, he has received numerous accolades, including an Academy Award and a Golden Globe...",https://en.wikipedia.org/w/index.php?title=Morgan_Freeman&oldid=1384898723,http://www.wikidata.org/entity/Q48337,
2,tt0111161,The Shawshank Redemption,1994,https://en.wikipedia.org/w/index.php?title=The_Shawshank_Redemption&oldid=1385870872,3,nm0348409,Bob Gunton,actor,1945-11-15,None,Santa Monica,"Robert Patrick Gunton Jr. (born November 15, 1945) is an American character actor of stage and screen. He is known for playing characters including Warden Samuel Norton in the 1994 prison drama Th...",https://en.wikipedia.org/w/index.php?title=Bob_Gunton&oldid=1362769678,http://www.wikidata.org/entity/Q352203,
3,tt0468569,The Dark Knight,2008,https://en.wikipedia.org/w/index.php?title=The_Dark_Knight&oldid=1384066085,1,nm0000288,Christian Bale,actor,1974-01-30,None,Haverfordwest,"Christian Charles Philip Bale (born 30 January 1974) is an English actor. Known for his versatility and physical transformations for his roles, he has been a leading man in films of several genres...",https://en.wikipedia.org/w/index.php?title=Christian_Bale&oldid=1383094254,http://www.wikidata.org/entity/Q45772,
4,tt0468569,The Dark Knight,2008,https://en.wikipedia.org/w/index.php?title=The_Dark_Knight&oldid=1384066085,2,nm0005132,Heath Ledger,actor,1979-04-04,2008-01-22,Perth,"Heath Andrew Ledger (4 April 1979 – 22 January 2008) was an Australian actor. Known for his versatility across independent and major studio films, his work consisted of 20 films in a variety of ge...",https://en.wikipedia.org/w/index.php?title=Heath_Ledger&oldid=1385796678,http://www.wikidata.org/entity/Q40572,
5,tt0468569,The Dark Knight,2008,https://en.wikipedia.org/w/index.php?title=The_Dark_Knight&oldid=1384066085,3,nm0001173,Aaron Eckhart,actor,1968-03-12,None,Cupertino,"Aaron Edward Eckhart (born March 12, 1968) is an American actor.\nBorn in Cupertino, California, Eckhart moved to the United Kingdom at an early age. He began his acting career by performing in sc...",https://en.wikipedia.org/w/index.php?title=Aaron_Eckhart&oldid=1384567512,http://www.wikidata.org/entity/Q192643,
6,tt1375666,Inception,2010,https://en.wikipedia.org/w/index.php?title=Inception&oldid=1385346213,1,nm0000138,Leonardo DiCaprio,actor,1974-11-11,None,Hollywood,"Leonardo Wilhelm DiCaprio (Italian: [diˈkaːprjo]; born November 11, 1974) is an American actor and film producer. Regarded as one of Hollywood’s defining leading men, his work spans character-driv...",https://en.wikipedia.org/w/index.php?title=Leonardo_DiCaprio&oldid=1385811153,http://www.wikidata.org/entity/Q38111,
7,tt1375666,Inception,2010,https://en.wikipedia.org/w/index.php?title=Inception&oldid=1385346213,2,nm0330687,Joseph Gordon-Levitt,actor,1981-02-17,None,Los Angeles,"Joseph Leonard Gordon-Levitt (born February 17, 1981) is an American actor and filmmaker. He has received various accolades, including nominations for the Golden Globe Award for Best Actor in a Mo...",https://en.wikipedia.org/w/index

,tconst,primaryTitle,startYear,genres,averageRating,numVotes,synopsis_section,synopsis,summary,synopsis_source_url,wikidata_url,principals_with_biography,enrichment_notes,lineage
0,tt0111161,The Shawshank Redemption,1994,Drama,9.3,3264541,Plot,"In 1947, Portland, Maine, banker Andy Dufresne arrives at Shawshank State Prison to serve two consecutive life sentences for murdering his wife and her lover. He is befriended by Ellis Boyd ""Red"" ...","The Shawshank Redemption is a 1994 American drama film written and directed by Frank Darabont, based on the 1982 Stephen King novella Rita Hayworth and Shawshank Redemption. The film tells the sto...",https://en.wikipedia.org/w/index.php?title=The_Shawshank_Redemption&oldid=1385870872,http://www.wikidata.org/entity/Q172241,3,,http://www.wikidata.org/entity/Q172241 https://en.wikipedia.org/w/index.php?title=The_Shawshank_Redemption&oldid=1385870872 https://en.wikipedia.org/w/index.php?title=Tim_Robbins&oldid=1384219365 ...
1,tt0468569,The Dark Knight,2008,"Crime,Thriller",9.1,3250365,Plot,"A gang of masked criminals rob a mafia-owned bank in Gotham City, betraying and killing each other until the sole survivor, the Joker, reveals himself as the mastermind and escapes with the money....","The Dark Knight is a 2008 superhero film directed by Christopher Nolan, who co-wrote the screenplay with his brother Jonathan. Based on the DC Comics superhero Batman, it is the sequel to Batman B...",https://en.wikipedia.org/w/index.php?title=The_Dark_Knight&oldid=1384066085,http://www.wikidata.org/entity/Q163872,3,,http://www.wikidata.org/entity/Q163872 https://en.wikipedia.org/w/index.php?title=The_Dark_Knight&oldid=1384066085 https://en.wikipedia.org/w/index.php?title=Christian_Bale&oldid=1383094254 https:...
2,tt1375666,Inception,2010,"Adventure,Sci-Fi,Thriller",8.8,2889326,Plot,Dom Cobb and Arthur are professional thieves who perform corporate espionage using experimental dream-sharing technology to infiltrate their targets' subconscious and extract information. Their la...,"Inception is a 2010 science fiction heist film written and directed by Christopher Nolan, who also produced it with his wife Emma Thomas. The film stars Leonardo DiCaprio as a professional thief w...",https://en.wikipedia.org/w/index.php?title=Inception&oldid=1385346213,http://www.wikidata.org/entity/Q25188,3,,http://www.wikidata.org/entity/Q25188 https://en.wikipedia.org/w/index.php?title=Inception&oldid=1385346213 https://en.wikipedia.org/w/index.php?title=Leonardo_DiCaprio&oldid=1385811153 https://en...
3,tt0137523,Fight Club,1999,"Crime,Drama,Thriller",8.8,2683389,Plot,"The unnamed Narrator works an office job as an insurance claims adjuster. He sees a doctor for his disordered sleep, complaining that he falls asleep unexpectedly and wakes up in unfamiliar places...","Fight Club is a 1999 American film directed by David Fincher starring Brad Pitt, Edward Norton, and Helena Bonham Carter. It is based on the 1996 novel Fight Club by Chuck Palahniuk. Norton plays ...",https://en.wikipedia.org/w/index.php?title=Fight_Club&oldid=1385277614,http://www.wikidata.org/entity/Q190050,3,,http://www.wikidata.org/entity/Q190050 https://en.wikipedia.org/w/index.php?title=Fight_Club&oldid=1385277614 https://en.wikipedia.org/w/index.php?title=Brad_Pitt&oldid=1385845504 https://en.wikip...
4,tt0816692,Interstellar,2014,"Adventure,Sci-Fi",8.7,2633912,Plot,"In the near future, humanity faces extinction due to dust storms and widespread crop blights. Joseph Cooper, a widowed former NASA test pilot, works as a farmer and raises his children, Murphy ""Mu...","</noinclude>\n\nInterstellar is a 2014 epic science fiction drama film directed by Christopher Nolan, who co-wrote the screenplay with his brother, Jonathan Nolan. It features an ensemble cast led...",https://en.wikipedia.org/w/index.php?title=Interstellar_(film)&oldid=1384598366,http://www.wikidata.org/entity/Q13417189,3,,http://www.wikidata.org/entity/Q13417189 https://en.wikipedia.org/w/i

## 9. Coverage metrics

In [10]:
coverage = spark.sql(f'''
SELECT COUNT(*) AS films,
       SUM(CASE WHEN wikidata_url IS NOT NULL THEN 1 ELSE 0 END) AS with_wikidata_item,
       SUM(CASE WHEN wikipedia_url IS NOT NULL THEN 1 ELSE 0 END) AS with_wikipedia_article,
       SUM(CASE WHEN synopsis IS NOT NULL THEN 1 ELSE 0 END) AS with_synopsis,
       SUM(CASE WHEN summary IS NOT NULL THEN 1 ELSE 0 END) AS with_summary,
       SUM(CASE WHEN principals_with_biography >= {PRINCIPALS_PER_FILM} THEN 1 ELSE 0 END) AS with_3_biographies,
       ROUND(AVG(principals_with_biography), 2) AS avg_biographies_per_film,
       SUM(CASE WHEN size(enrichment_notes) > 0 THEN 1 ELSE 0 END) AS films_with_notes
FROM enriched_films
''').toPandas().T.rename(columns={0: "value"})
count_rows = coverage.index != "avg_biographies_per_film"
coverage.loc[count_rows, "pct_of_films"] = (
    100 * coverage.loc[count_rows, "value"] / coverage.loc["films", "value"]
).round(1) if coverage.loc["films", "value"] else None
display(coverage)  # Enrichment coverage: how many films got a synopsis and three principal biographies
coverage.to_csv(OUTPUT_DIR / "enrichment_coverage.csv")

note_counts = export(  # How often each kind of missing-information note was added
    spark.sql('''
    SELECT 'film' AS level, note, COUNT(*) AS occurrences
    FROM enriched_films LATERAL VIEW explode(enrichment_notes) n AS note GROUP BY note
    UNION ALL
    SELECT 'principal' AS level, note, COUNT(*) AS occurrences
    FROM selected_principals LATERAL VIEW explode(notes) n AS note GROUP BY note
    ORDER BY level, occurrences DESC
    '''),
    "enrichment_note_counts.csv",
)

,value,pct_of_films
films,2090.00,100.0
with_wikidata_item,2090.00,100.0
with_wikipedia_article,2090.00,100.0
with_synopsis,2088.00,99.9
with_summary,2090.00,100.0
with_3_biographies,2069.00,99.0
avg_biographies_per_film,2.99,NaN
films_with_notes,23.00,1.1


,level,note,occurrences
0,film,Only 2 of 3 selected principals have a Wikipedia biography,16
1,film,Only 1 of 3 selected principals have a Wikipedia biography,4
2,film,Article has no Plot/Synopsis section; only the introduction is provided as summary,2
3,film,Only 0 of 3 selected principals have a Wikipedia biography,1
4,principal,No Wikidata item has this IMDb ID,18
5,principal,No English Wikipedia article,9
6,principal,Several Wikidata items share this IMDb ID; the one with an English article was used,3


### Example of an enriched record

In [11]:
example = enriched_films.orderBy(F.desc("numVotes")).limit(1).toJSON().collect()
print(json.dumps(json.loads(example[0]), indent=2, ensure_ascii=False)[:6000])

{
  "tconst": "tt0111161",
  "primaryTitle": "The Shawshank Redemption",
  "originalTitle": "The Shawshank Redemption",
  "startYear": 1994,
  "runtimeMinutes": 142,
  "genres": [
    "Drama"
  ],
  "genre_year_ranks": [
    {
      "genre": "Drama",
      "rank": 1
    }
  ],
  "averageRating": 9.3,
  "numVotes": 3264541,
  "wikidata_url": "http://www.wikidata.org/entity/Q172241",
  "wikipedia_url": "https://en.wikipedia.org/wiki/The_Shawshank_Redemption",
  "synopsis_source_url": "https://en.wikipedia.org/w/index.php?title=The_Shawshank_Redemption&oldid=1385870872",
  "synopsis_section": "Plot",
  "synopsis": "In 1947, Portland, Maine, banker Andy Dufresne arrives at Shawshank State Prison to serve two consecutive life sentences for murdering his wife and her lover. He is befriended by Ellis Boyd \"Red\" Redding, a contraband smuggler serving a life sentence, who procures for him a rock hammer and a large poster of Rita Hayworth. Assigned to work in the prison laundry, Andy is freque